In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\karasoo1\Downloads\SDM_RF_Raw_CV_RawScores.csv")
print(f"Loaded {len(df):,} rows  |  columns: {list(df.columns)}")
print(df[['set', 'presence']].value_counts().sort_index(), "\n")

# ── Per-fold, per-set metrics ─────────────────────────────────────────────────
records = []

for (fold_id, set_type), grp in df.groupby(['fold_id', 'set']):
    y_true = grp['presence'].values.astype(int)
    y_prob = grp['prob'].values.astype(float)

    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos

    # ── AUC ───────────────────────────────────────────────────────────────────
    auc = roc_auc_score(y_true, y_prob)

    # ── Log-loss ──────────────────────────────────────────────────────────────
    ll = log_loss(y_true, y_prob)

    # ── Brier score & Brier Skill Score ───────────────────────────────────────
    brier = brier_score_loss(y_true, y_prob)
    prevalence = n_pos / len(y_true)
    brier_ref  = prevalence * (1 - prevalence)
    bss = 1 - brier / (brier_ref + 1e-9)

    # ── Calibration (mean predicted prob per class) ───────────────────────────
    mean_prob_pos = y_prob[y_true == 1].mean()
    mean_prob_neg = y_prob[y_true == 0].mean()

    # ── Cohen's d (discrimination) ────────────────────────────────────────────
    prob_pos = y_prob[y_true == 1]
    prob_neg = y_prob[y_true == 0]
    pooled_sd = np.sqrt(
        ((n_pos - 1) * prob_pos.std(ddof=1)**2 +
         (n_neg - 1) * prob_neg.std(ddof=1)**2)
        / (n_pos + n_neg - 2 + 1e-9)
    )
    cohens_d = (prob_pos.mean() - prob_neg.mean()) / (pooled_sd + 1e-9)

    records.append({
        'fold_id':           fold_id,
        'set':               set_type,
        'n_presence':        int(n_pos),
        'n_absence':         int(n_neg),
        'AUC':               auc,
        'Log_Loss':          ll,
        'Brier_Score':       brier,
        'Brier_Skill_Score': bss,
        'Mean_Prob_Presence': mean_prob_pos,
        'Mean_Prob_Absence':  mean_prob_neg,
        'Cohens_D':          cohens_d,
    })

metrics = pd.DataFrame(records).sort_values(['set', 'fold_id'])

# ── Print per-fold results ────────────────────────────────────────────────────
print("=" * 70)
print("PER-FOLD METRICS")
print("=" * 70)
print(metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ── Summary: both train and test ─────────────────────────────────────────────
metric_cols = ['AUC', 'Log_Loss', 'Brier_Score', 'Brier_Skill_Score',
               'Mean_Prob_Presence', 'Mean_Prob_Absence', 'Cohens_D']

summary_rows = []
for set_type in ['test', 'train']:
    grp = metrics[metrics['set'] == set_type][metric_cols]
    for col in metric_cols:
        summary_rows.append({
            'Set':    set_type,
            'Metric': col,
            'Mean':   grp[col].mean(),
            'SD':     grp[col].std(ddof=1),
            'Min':    grp[col].min(),
            'Max':    grp[col].max(),
        })

summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("CV SUMMARY  (n=5 folds, both train and test)")
print("=" * 70)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Also produce a wide version convenient for copy-pasting into a paper
wide = summary.pivot(index='Metric', columns='Set', values=['Mean', 'SD'])
wide.columns = [f"{stat}_{s}" for stat, s in wide.columns]
wide = wide.reindex(metric_cols)  # preserve logical row order
wide['Test_Mean±SD'] = wide.apply(
    lambda r: f"{r['Mean_test']:.3f} ± {r['SD_test']:.3f}", axis=1)
wide['Train_Mean±SD'] = wide.apply(
    lambda r: f"{r['Mean_train']:.3f} ± {r['SD_train']:.3f}", axis=1)

print("\n" + "=" * 70)
print("PAPER-READY TABLE  (Mean ± SD across 5 folds)")
print("=" * 70)
print(wide[['Test_Mean±SD', 'Train_Mean±SD']].to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
out_dir = r"C:\Users\karasoo1\Downloads"
metrics.to_csv(f"{out_dir}\\SDM_RF_CV_ProbMetrics.csv",  index=False)
summary.to_csv(f"{out_dir}\\SDM_RF_CV_ProbSummary.csv",  index=False)
wide[['Test_Mean±SD', 'Train_Mean±SD']].to_csv(
    f"{out_dir}\\SDM_RF_CV_ProbSummary_Wide.csv")

print(f"\nSaved:")
print(f"  {out_dir}\\SDM_RF_CV_ProbMetrics.csv       (per-fold, all sets)")
print(f"  {out_dir}\\SDM_RF_CV_ProbSummary.csv        (long format)")
print(f"  {out_dir}\\SDM_RF_CV_ProbSummary_Wide.csv   (paper-ready)")

Loaded 200,570 rows  |  columns: ['fold_id', 'set', 'presence', 'prob']
set    presence
test   0           19995
       1           20119
train  0           79980
       1           80476
Name: count, dtype: int64 

PER-FOLD METRICS
 fold_id   set  n_presence  n_absence    AUC  Log_Loss  Brier_Score  Brier_Skill_Score  Mean_Prob_Presence  Mean_Prob_Absence  Cohens_D
  0.0000  test        4019       3819 0.8845    0.4266       0.1387             0.4448              0.7081             0.3033    1.8063
  1.0000  test        4009       3211 0.8851    0.4276       0.1392             0.4361              0.7055             0.2930    1.7932
  2.0000  test        4022       4322 0.8922    0.4140       0.1322             0.4707              0.7014             0.2872    1.9217
  3.0000  test        4045       4102 0.8937    0.4165       0.1329             0.4685              0.7104             0.3034    1.9207
  4.0000  test        4024       4541 0.8865    0.4235       0.1357             0.4550 